# Projeto 1 - Pipeline SOTA de Grafos de Conhecimento Clínicos (MC896)
**Disciplina:** Processamento de Línguas Naturais (MC896 / 2s2026) - Unicamp  
**Professor:** André Santanchè  
**Equipe:** Caio, Fábio Luiz, Gustavo Sousa, T.H. de Camargo J., Wallysson

---
## 1. Visão Geral da Arquitetura em Duas Camadas
Este notebook demonstra a extração determinística clássica (sem modelos de linguagem neurais) dos casos clínicos do **MultiCaRe** para um grafo de propriedades em duas camadas:
1. **Camada Episódica (Local por Paciente):** Instâncias de observações, exames, dosagens, valores com status de asserção e evidência textual.
2. **Camada Canônica (Conceitual Global):** Nós MeSH/LOINC/RxNorm indexados, ligados via arestas `INSTANCE_OF`.
3. **Camada Temporal:** Grafo direcionado acíclico (DAG) através de arestas `PRECEDES` baseadas em âncoras TimeML.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import json

# Ajusta path para importar o pipeline dinamicamente de qualquer CWD
current_path = Path(".").resolve()
if (current_path / "project1").exists():
    ROOT_DIR = current_path
elif (current_path.parent / "project1").exists():
    ROOT_DIR = current_path.parent
elif (current_path.parent.parent / "project1").exists():
    ROOT_DIR = current_path.parent.parent
else:
    ROOT_DIR = current_path
sys.path.insert(0, str(ROOT_DIR))

from project1.src.preprocessor import ClinicalPreprocessor
from project1.src.entity_matcher import ClinicalEntityMatcher
from project1.src.assertion import ClinicalAssertionAnalyzer
from project1.src.quant_extractor import ClinicalQuantExtractor
from project1.src.timeline_dag import ClinicalTimelineExtractor
from project1.src.graph_builder import ClinicalGraphBuilder
from project1.src.evaluate_mesh import run_benchmark

print("Todos os módulos SOTA carregados com sucesso!")

## 2. Pré-processamento & Segmentação com Proteção Léxica
Demonstração de proteção contra a quebra de sentenças em pontos decimais (`850.5 U/L`, `0.8 cm`) e siglas médicas (`Fig. 1`, `POD 7`).

In [ ]:
matcher = ClinicalEntityMatcher()
test_phrase = "Patient presented with acute pancreatitis, severe epigastric pain, and clear breath sounds after cold-knife conization."
matches = matcher.match_entities_in_sentence(test_phrase)
pd.DataFrame(matches)[["matched_text", "canonical", "category", "type"]]

## 3. Aho-Corasick com Token Boundary e Longest-Match-First
Elimina falsos positivos em substrings (ex.: `cold` em `cold-knife`) e prioriza termos compostos sobre termos redundantes (`acute pancreatitis` sobrepõe `pancreatitis`).

In [ ]:
matcher = ClinicalEntityMatcher(gazetteer_path="../project1/data/vocabularies/clinical_gazetteer.json")
test_phrase = "Patient presented with acute pancreatitis, severe epigastric pain, and clear breath sounds after cold-knife conization."
matches = matcher.match_entities_in_sentence(test_phrase)
pd.DataFrame(matches)[["matched_text", "canonical", "category", "type"]]

nodes_df = pd.read_csv(ROOT_DIR / "project1/data/output/nodes.csv")
edges_df = pd.read_csv(ROOT_DIR / "project1/data/output/edges.csv")
print(f"Total de Nós: {len(nodes_df)}")
print(f"Total de Arestas: {len(edges_df)}")
print("\nTop 5 Nós:")
display(nodes_df.head())
print("\nTop 5 Arestas:")
display(edges_df.head())

In [ ]:
with open(ROOT_DIR / "project1/data/output/benchmark_results.json", "r", encoding="utf-8") as f:
    bench = json.load(f)
bench_df = pd.DataFrame([
    {"Métrica": k, "Baseline Ingênua": bench["naive_baseline"][k], "Pipeline SOTA": bench["sota_pipeline"][k]}
    for k in ["precision", "recall", "f1_score", "jaccard", "recall_at_10"]
])
display(bench_df)

## 5. Extração de Resultados Laboratoriais e Dosagens Farmacológicas
Isolamento sintagmático de listas coordenadas e captura de faixas de referência.

In [ ]:
quant = ClinicalQuantExtractor()
lab_text = "Hemoglobin was 9.2 g/dL, WBC 14,500 /mcL, and platelets 85,000 /mcL with elevated lipase (850 U/L, reference range 10-140 U/L)."
labs = quant.extract_lab_results(lab_text)
pd.DataFrame(labs)[["exam_name", "value", "unit", "reference_low", "reference_high", "interpretation"]]

## 6. Processamento do Dataset & Estrutura do Grafo Gerado
Carrega as tabelas finais exportadas `nodes.csv` e `edges.csv`.

In [ ]:
nodes_df = pd.read_csv("../project1/data/output/nodes.csv")
edges_df = pd.read_csv("../project1/data/output/edges.csv")
print(f"Total de Nós: {len(nodes_df)}")
print(f"Total de Arestas: {len(edges_df)}")
print("\nTop 5 Nós:")
display(nodes_df.head())
print("\nTop 5 Arestas:")
display(edges_df.head())

## 7. Benchmark Quantitativo: Baseline Ingênua vs. Pipeline SOTA
Comparação formal contra o padrão-ouro humano indexado na National Library of Medicine (`metadata.csv`).

In [ ]:
with open("../project1/data/output/benchmark_results.json", "r", encoding="utf-8") as f:
    bench = json.load(f)
bench_df = pd.DataFrame([
    {"Métrica": k, "Baseline Ingênua": bench["naive_baseline"][k], "Pipeline SOTA": bench["sota_pipeline"][k]}
    for k in ["precision", "recall", "f1_score", "jaccard", "recall_at_10"]
])
display(bench_df)